# 01 · 2025 언론수용자 조사 EDA — 순진한 베이스라인 → 문제 진단

> 데이터: `data/raw/audience/2025/2025_언론수용자조사_데이터.sav` (N=6,000 · 223변수 · EUC-KR · 가중치 `WT`)
> 변수 정본: [`docs/design/2025-sav-variables.md`](../docs/design/2025-sav-variables.md) · 명세: [`docs/design/data-spec.md`](../docs/design/data-spec.md) §5

## 목적
"뉴스 건강지수" 4차원(**신뢰·다양성·검증습관·회피**) 후보 변수의 **분포·결측·특수코드**를 점검하고,
**전처리 前 순진한 베이스라인 지수**를 일부러 산출해 문제를 노출 → 진단 → 차원 최종 확정/축소를 결정한다.

## 설계 원칙
- **경량 우선**: pandas·기초통계만. 무거운 transformer·시각화 라이브러리 배제(텍스트·표 출력 중심).
- **Before/After 스토리텔링**: 순진한 결과 → 문제 진단 → 개선 Δ.
- **분석 수치는 KPF 원자료 재검증 전까지 보고서 직접 인용 금지**(본 노트북은 내부 설계 근거).

## TL;DR (본 노트북 결론)
| 차원 | 가용성 | 결정 |
|------|:----:|------|
| **신뢰** | ★★★ 배터리 결측·특수코드 0% | **코어 채택**(역코딩·요인구조 정제 필요) |
| **다양성** | ★★ 이용매체 21문항+Q84 | **코어 채택**(가중·연령비닝·9998 분리 필수) |
| 검증습관 | △ 구조적 결측 30~74%·약한 상관 | **보조 지표로 강등**(코어 제외) |
| 회피 | △ Q84==9998 4.35%뿐 | **이진 행동 플래그로 강등**(차원 아님) |

→ **신뢰·다양성 2개 코어 차원** + 검증·회피 보조 플래그 (data-spec §5 옵션 b 채택).

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

# ── 경로 해석: notebooks/에서 실행되든 repo 루트에서 실행되든 동작 ──
def find_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "data/raw/audience/2025").exists():
            return cand
    raise FileNotFoundError("data/raw/audience/2025 를 찾지 못했습니다")

ROOT = find_root()
SAV = ROOT / "data/raw/audience/2025/2025_언론수용자조사_데이터.sav"
PARQUET = ROOT / "data/interim/2025_audience.parquet"
META_JSON = ROOT / "data/interim/_2025_sav_meta.json"

# 결측 외 특수코드(코드북): 9997=기타, 9998=해당없음/이용안함, 9999=모름·무응답
SPECIAL = [9997, 9998, 9999]
pd.set_option("display.max_columns", 40)
print("ROOT:", ROOT)

ROOT: C:\Users\kik32\workspace\Dacon\Media-Statistics-Analysis-and-Utilization-Competition


## 1. 1회 로딩 → `data/interim` parquet 캐시

`.sav`(EUC-KR)는 `pyreadstat`로 **딱 한 번** 읽어 parquet으로 캐시한다. 이후 셀·재실행은 parquet에서 즉시 로드(반복 풀로딩 금지).
parquet은 `.gitignore` 대상이며 `.sav`에서 항상 재생성 가능.

In [2]:
def load_2025() -> pd.DataFrame:
    """2025 .sav를 parquet 캐시 경유로 로딩(없으면 생성)."""
    if PARQUET.exists():
        return pd.read_parquet(PARQUET), "parquet(cache)"
    df, _ = pyreadstat.read_sav(str(SAV), encoding="euc-kr")
    PARQUET.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(PARQUET)
    return df, "sav(최초 로딩→캐시 생성)"

df, _src = load_2025()
labels = {v["name"]: v["label"] for v in json.loads(META_JSON.read_text(encoding="utf-8"))["vars"]}
print(f"로딩 소스: {_src}")
print(f"shape: {df.shape[0]:,}행 × {df.shape[1]}변수")

로딩 소스: parquet(cache)
shape: 6,000행 × 223변수


## 2. 데이터 개관 — 가중치와 표본 연령

핵심 두 가지: ① 가중치 `WT`의 분산이 크다(가중 필요), ② **표본이 고령으로 치우쳐** 있다(중앙 연령 53세) → 모집단 추정에 가중·연령보정이 필수임을 시사.

In [3]:
print("[WT 가중치] (합 = 가중 후 모집단 규모)")
print(df["WT"].describe().round(3).to_string())
print(f"WT 합계: {df['WT'].sum():,.0f}   |   WT 표준편차 {df['WT'].std():.3f}, 최대 {df['WT'].max():.2f}배\n")

print("[DQ3 연령] — 연속값(만 나이). 분석 시 연령대 비닝 직접 수행 필요")
print(df["DQ3"].describe().round(1).to_string())
print(f"\n→ 표본 중앙연령 {df['DQ3'].median():.0f}세: 디지털 매체 과소대표 가능 → 가중·연령보정 동기")

[WT 가중치] (합 = 가중 후 모집단 규모)
count    6000.000
mean        1.000
std         0.767
min         0.088
25%         0.494
50%         0.755
75%         1.266
max         6.924
WT 합계: 6,000   |   WT 표준편차 0.767, 최대 6.92배

[DQ3 연령] — 연속값(만 나이). 분석 시 연령대 비닝 직접 수행 필요
count    6000.0
mean       51.3
std        16.2
min        19.0
25%        37.0
50%        53.0
75%        64.0
max        97.0

→ 표본 중앙연령 53세: 디지털 매체 과소대표 가능 → 가중·연령보정 동기


## 3. 4차원 후보 변수 정의

data-spec §5 / 변수 정본을 코드로 고정한다. 신뢰 배터리는 **양성 키(높을수록 신뢰↑)**와 **역키 Q91(심각도, 높을수록 부정)**을 구분한다.

In [4]:
# 신뢰 — 양성 키(5점, 높을수록 긍정)
TRUST_POS = ([f"Q85_{i}" for i in range(1,6)] + [f"Q86_{i}" for i in range(1,8)]
             + ["Q87_1","Q87_2"] + [f"Q88_{i}" for i in range(1,4)] + ["Q93"]
             + [f"Q94_{i}" for i in range(1,5)] + [f"Q95_{i}" for i in range(1,11)] + ["Q96"])
# 신뢰 — 역키(보도 문제점 심각도: 높을수록 부정 → 역코딩 필요)
TRUST_REV = [f"Q91_{i}" for i in range(1,9)]
# 다양성 — 매체/뉴스 이용여부 게이트 21문항(1=이용, 2=비이용)
GATE = ["Q1","Q6","Q8","Q10","Q13","Q17","Q20","Q36","Q39","Q43","Q46",
        "Q50","Q53","Q58","Q61","Q64","Q67","Q70","Q73","Q76","Q78"]
# 검증습관 — 직접문항 없음. 프록시(능동 구독/검색/출처인지)
VERIFY = ["Q93","Q34_3","Q56_2","Q34_5","Q56_4"]
# 회피 — 직접문항 없음. Q84 주이용경로 9998(=이용 안함)
AVOID_CODE = 9998
print(f"신뢰 양성 {len(TRUST_POS)} · 역키 {len(TRUST_REV)} · 다양성 게이트 {len(GATE)} · 검증 프록시 {len(VERIFY)}")

신뢰 양성 33 · 역키 8 · 다양성 게이트 21 · 검증 프록시 5


## 4. 기초 분포 · 결측 · 특수코드 비율

각 변수의 결측%, 특수코드(9997/9998/9999)%, 정상값 평균을 본다. **신뢰 배터리는 매우 깨끗**한 반면 **검증 프록시는 구조적(게이트) 결측이 심각**함이 드러난다.

In [5]:
def special_profile(cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in cols:
        s = df[c]; n = len(s)
        na = s.isna().sum(); sp = int(s.isin(SPECIAL).sum())
        valid = s[~s.isna() & ~s.isin(SPECIAL)]
        rows.append([c, round(100*na/n,1), round(100*sp/n,1),
                     round(valid.mean(),3) if len(valid) else np.nan, len(valid),
                     labels.get(c,"")[:34]])
    return pd.DataFrame(rows, columns=["var","결측%","특수%","정상평균","유효N","문항"])

print("■ 신뢰 양성 배터리(앞 8개) — 결측·특수코드 거의 0%")
display(special_profile(TRUST_POS[:8]))
print("■ 신뢰 역키 Q91_* (심각도) — 깨끗하나 방향이 반대")
display(special_profile(TRUST_REV))

■ 신뢰 양성 배터리(앞 8개) — 결측·특수코드 거의 0%


,var,결측%,특수%,정상평균,유효N,문항
0,Q85_1,0.0,0.0,3.205,6000,문85_1. 언론에 대한 인식: 우리나라 언론은 공정하다
1,Q85_2,0.0,0.0,3.538,6000,문85_2. 언론에 대한 인식: 우리나라 언론은 전문적이다
2,Q85_3,0.0,0.0,3.354,6000,문85_3. 언론에 대한 인식: 우리나라 언론은 정확하다
3,Q85_4,0.0,0.0,3.588,6000,문85_4. 언론에 대한 인식: 우리나라는 언론활동이 자유롭다
4,Q85_5,0.0,0.0,3.701,6000,문85_5. 언론에 대한 인식: 우리나라 언론은 영향력이 있다
5,Q86_1,0.0,0.0,3.430,6000,문86_1. 언론 역할별 수행 정도에 대한 인식: 사회 현안에
6,Q86_2,0.0,0.0,3.421,6000,문86_2. 언론 역할별 수행 정도에 대한 인식: 사회 현안에
7,Q86_3,0.0,0.0,3.304,6000,문86_3. 언론 역할별 수행 정도에 대한 인식: 사회 현안에


■ 신뢰 역키 Q91_* (심각도) — 깨끗하나 방향이 반대


,var,결측%,특수%,정상평균,유효N,문항
0,Q91_1,0.0,0.0,3.408,6000,문91_1. 뉴스/정보 관련 문제점: 언론사의 오보
1,Q91_2,0.0,0.0,3.626,6000,문91_2. 뉴스/정보 관련 문제점: 낚시성 기사
2,Q91_3,0.0,0.0,3.592,6000,문91_3. 뉴스/정보 관련 문제점: 어뷰징 기사
3,Q91_4,0.0,0.0,3.592,6000,문91_4. 뉴스/정보 관련 문제점: 편파적 기사
4,Q91_5,0.0,0.0,3.464,6000,문91_5. 뉴스/정보 관련 문제점: 광고성 기사
5,Q91_6,0.0,0.0,3.524,6000,문91_6. 뉴스/정보 관련 문제점: 언론사의 자사 이기주의적
6,Q91_7,0.0,0.0,3.540,6000,문91_7. 뉴스/정보 관련 문제점: 받아쓰기식 기사
7,Q91_8,0.0,0.0,3.586,6000,문91_8. 뉴스/정보 관련 문제점: 허위조작정보(가짜뉴스)


In [6]:
print("■ 검증습관 프록시 — 게이트로 인한 구조적 결측(Q56_*는 73.7%)")
display(special_profile(VERIFY))

print("■ 다양성 게이트 21문항 — 이용/비이용 분포 (Q8은 Q6 하위 게이트라 95.5% 결측)")
gate_tbl = pd.DataFrame({
    "var": GATE,
    "이용(1)%": [round(100*(df[c]==1).mean(),1) for c in GATE],
    "비이용(2)%": [round(100*(df[c]==2).mean(),1) for c in GATE],
    "결측%": [round(100*df[c].isna().mean(),1) for c in GATE],
    "문항": [labels.get(c,"")[:24] for c in GATE],
})
display(gate_tbl)

■ 검증습관 프록시 — 게이트로 인한 구조적 결측(Q56_*는 73.7%)


,var,결측%,특수%,정상평균,유효N,문항
0,Q93,30.1,0.0,2.978,4194,문93. 온라인 뉴스 작성/제공 언론사 인지 여부
1,Q34_3,35.5,0.0,2.622,3871,문34_3. 인터넷 포털 뉴스 이용 방법_인터넷 포털에서 특정
2,Q56_2,73.7,0.0,2.821,1580,문56_2. 온라인 동영상 플랫폼 뉴스 이용 방법_특정 언론사
3,Q34_5,35.5,0.0,3.020,3871,문34_5. 인터넷 포털 뉴스 이용 방법_인터넷 포털에서 필요
4,Q56_4,73.7,0.0,3.015,1580,문56_4. 온라인 동영상 플랫폼 뉴스 이용 방법_필요한 정보


■ 다양성 게이트 21문항 — 이용/비이용 분포 (Q8은 Q6 하위 게이트라 95.5% 결측)


,var,이용(1)%,비이용(2)%,결측%,문항
0,Q1,7.7,92.3,0.0,문1. 지난 일주일 간 종이신문 열독 여부
1,Q6,4.4,95.6,0.0,문6. 지난 1주일 간 종이잡지 열독 여부
2,Q8,1.8,2.6,95.6,문8. 지난 일주일 간 뉴스/시사 잡지 열독
3,Q10,94.2,5.8,0.0,문10. 지난 일주일 간 TV 프로그램 시청
4,Q13,81.3,18.7,0.0,문13. 지난 일주일 간 TV 뉴스/시사 프
5,Q17,14.3,85.7,0.0,문17. 지난 일주일 간 라디오 프로그램 청
6,Q20,7.4,92.6,0.0,문20. 지난 일주일 간 라디오 뉴스/시사
7,Q36,90.9,9.1,0.0,문36. 지난 일주일 간 메신저 서비스 이용
8,Q39,12.1,87.9,0.0,문39. 지난 일주일 간 메신저 서비스 뉴스
9,Q43,42.0,58.0,0.0,문43. 지난 일주일 간 SNS 이용 여부


In [7]:
# 회피 후보 Q84(주 이용경로) 분포
q84 = df["Q84"]
avoid_share = 100*(q84==AVOID_CODE).mean()
print(f"[회피] Q84==9998(주경로로 뉴스 이용 안함) = {avoid_share:.2f}%  (n={int((q84==AVOID_CODE).sum())})")
print("→ 시장조사 '의도적 뉴스회피 72%(2024)'와 큰 격차: 본조사엔 직접 회피문항 부재\n")
print("[Q84 주이용경로 상위] (9998 제외, %)")
print((100*q84[q84!=AVOID_CODE].value_counts(normalize=True)).round(1).head(6).to_string())

[회피] Q84==9998(주경로로 뉴스 이용 안함) = 4.35%  (n=261)
→ 시장조사 '의도적 뉴스회피 72%(2024)'와 큰 격차: 본조사엔 직접 회피문항 부재

[Q84 주이용경로 상위] (9998 제외, %)
Q84
2.0     52.3
5.0     32.3
13.0     5.0
10.0     2.3
8.0      1.4
1.0      1.3


## 5. 순진한 베이스라인 지수 (전처리 前)

가중 미적용·결측 미처리·특수코드 미분리·역코딩 미적용·연령 미비닝 상태로 4차원을 **그대로** 평균낸다.
의도적으로 왜곡을 만들어 §6에서 진단한다.

In [8]:
# 순진한 신뢰: 양성+역키를 구분 없이 단순평균(역코딩 미적용)
naive_trust = df[TRUST_POS + TRUST_REV].mean(axis=1)
# 순진한 다양성: 게이트 1 카운트(NaN을 비이용처럼 취급)
naive_div = (df[GATE]==1).sum(axis=1)
# 순진한 회피: Q84를 '숫자'로 그대로 평균 (9998 포함)
naive_avoid_mean = df["Q84"].mean()

print(f"순진 신뢰 평균      : {naive_trust.mean():.3f}  (역키 Q91 섞임)")
print(f"순진 다양성 평균(개수): {naive_div.mean():.2f}")
print(f"순진 회피(Q84 raw평균): {naive_avoid_mean:.1f}  ← 경로코드 평균은 의미 없음(9998이 끌어올림)")
print("\n→ 세 줄 모두 '그럴듯한 숫자'지만 아래 진단처럼 해석 불가/왜곡 상태")

순진 신뢰 평균      : 3.274  (역키 Q91 섞임)
순진 다양성 평균(개수): 6.18
순진 회피(Q84 raw평균): 439.0  ← 경로코드 평균은 의미 없음(9998이 끌어올림)

→ 세 줄 모두 '그럴듯한 숫자'지만 아래 진단처럼 해석 불가/왜곡 상태


## 6. 문제 진단 (Before → After)

### 6.1 역코딩 미적용 — 신뢰 지수 +0.21 거품
Q91(보도 문제점 *심각도*)은 높을수록 **부정**인데 순진 평균은 이를 신뢰에 그대로 더해 점수를 부풀린다.

In [9]:
clean = df[TRUST_POS + TRUST_REV].mask(df[TRUST_POS + TRUST_REV].isin(SPECIAL)).astype(float)
for c in TRUST_REV:
    clean[c] = 6 - clean[c]              # 5점 역코딩
clean_trust = clean.mean(axis=1)
print(f"Before(순진)={naive_trust.mean():.3f}  →  After(역코딩+특수제거)={clean_trust.mean():.3f}"
      f"  Δ={naive_trust.mean()-clean_trust.mean():+.3f}")

Before(순진)=3.274  →  After(역코딩+특수제거)=3.061  Δ=+0.213


### 6.2 특수코드 9998 실값 오인 — 행동변수에서 치명적
신뢰 배터리엔 특수코드가 없지만, **Q84·게이트 행동변수**에선 9998을 실값으로 쓰면 무의미해진다(평균 439).
다양성도 NaN을 '비이용'으로 뭉뚱그리면 분모가 흔들린다.

In [10]:
print(f"Q84 raw 평균(9998 포함) = {df['Q84'].mean():.1f}  → 9998 제외해야 경로 분포로 해석 가능")
# 다양성: NaN을 명시적으로 비이용 처리할지 vs 분모에서 제외할지에 따라 값이 달라짐
div_naive = (df[GATE]==1).sum(axis=1).mean()
div_strict = (df[GATE]==1).sum(axis=1)[df[GATE].notna().all(axis=1)].mean()
print(f"다양성 평균: NaN=비이용 취급 {div_naive:.2f}  vs  완전응답자만 {div_strict:.2f}")

Q84 raw 평균(9998 포함) = 439.0  → 9998 제외해야 경로 분포로 해석 가능
다양성 평균: NaN=비이용 취급 6.18  vs  완전응답자만 9.84


### 6.3 가중치 미적용 — 디지털 매체 이용률 과소추정
표본 고령 편중 탓에 **숏폼·온라인동영상 등 디지털 매체 이용률이 비가중에서 과소**추정된다.

In [11]:
w = df["WT"]
def wpct(mask): return 100*np.average(mask.astype(float), weights=w)
rows = []
for c, nm in [("Q1","종이신문"),("Q10","TV"),("Q50","온라인동영상"),("Q58","숏폼"),("Q70","생성형AI")]:
    u = 100*(df[c]==1).mean(); g = wpct(df[c]==1)
    rows.append([nm, round(u,1), round(g,1), round(g-u,1)])
display(pd.DataFrame(rows, columns=["매체","비가중%","가중%","Δ(가중-비가중)"]))
print(f"다양성 평균: 비가중 {naive_div.mean():.2f} → 가중 {np.average(naive_div,weights=w):.2f}")

,매체,비가중%,가중%,Δ(가중-비가중)
0,종이신문,7.7,8.4,0.7
1,TV,94.2,94.1,-0.1
2,온라인동영상,77.1,79.8,2.7
3,숏폼,56.2,59.2,2.9
4,생성형AI,19.3,20.7,1.3


다양성 평균: 비가중 6.18 → 가중 6.37


### 6.4 연령 미비닝 — 다양성의 강한 연령 구배 소실
신뢰는 연령과 거의 무상관(r≈0.07)이라 연속 처리해도 큰 손실이 없지만, **다양성은 연령대별로 가파르게 감소**(19–29세 7.5 → 70+ 4.0)해 비닝이 필수다.

In [12]:
labs = ["19-29","30-39","40-49","50-59","60-69","70+"]
band = pd.cut(df["DQ3"], [18,29,39,49,59,69,200], labels=labs)
t = pd.DataFrame({"band":band, "div":naive_div, "trust":clean_trust, "w":w}).dropna(subset=["band"])
gd = t.groupby("band", observed=True).apply(lambda x: np.average(x["div"], weights=x["w"]), include_groups=False)
gt = t.dropna(subset=["trust"]).groupby("band", observed=True).apply(lambda x: np.average(x["trust"], weights=x["w"]), include_groups=False)
display(pd.DataFrame({"다양성(가중)":gd.round(2), "신뢰(가중)":gt.round(3)}))
r = np.corrcoef(df["DQ3"][clean_trust.notna()], clean_trust[clean_trust.notna()])[0,1]
print(f"연속연령-신뢰 피어슨 r={r:.3f} (신뢰는 평탄) / 다양성은 연령대 단조 감소 → 비닝 필요")

,다양성(가중),신뢰(가중)
band,,
19-29,7.54,3.078
30-39,7.76,3.070
40-49,7.28,3.060
50-59,6.39,3.078
60-69,5.29,3.075
70+,3.97,3.143


연속연령-신뢰 피어슨 r=0.071 (신뢰는 평탄) / 다양성은 연령대 단조 감소 → 비닝 필요


### 6.5 검증·회피 프록시 타당도 — 약함
검증 프록시는 **구조적 결측이 크고**(Q56_* 73.7%) 신뢰와의 상관도 약하다(r 0.15~0.23).
회피 프록시(Q84==9998)는 4.35%로 작지만 방향성은 타당(회피군의 다양성·신뢰가 낮음).

In [13]:
print("[검증 프록시] 유효%·신뢰와의 상관")
for c in VERIFY:
    sub = df[c].mask(df[c].isin(SPECIAL)); m = sub.notna() & clean_trust.notna()
    r = np.corrcoef(sub[m], clean_trust[m])[0,1] if m.sum()>30 else np.nan
    print(f"  {c:6s} 유효%={100*sub.notna().mean():5.1f}  r(신뢰)={r:+.3f}")

avoid = (df["Q84"]==AVOID_CODE)
m = clean_trust.notna()
print(f"\n[회피 프록시] Q84==9998 = {100*avoid.mean():.2f}%")
print(f"  다양성: 회피군 {naive_div[avoid].mean():.2f} vs 비회피군 {naive_div[~avoid].mean():.2f}")
print(f"  신뢰  : 회피군 {clean_trust[avoid & m].mean():.3f} vs 비회피군 {clean_trust[~avoid & m].mean():.3f}")

[검증 프록시] 유효%·신뢰와의 상관
  Q93    유효%= 69.9  r(신뢰)=+0.221
  Q34_3  유효%= 64.5  r(신뢰)=+0.155
  Q56_2  유효%= 26.3  r(신뢰)=+0.179
  Q34_5  유효%= 64.5  r(신뢰)=+0.149
  Q56_4  유효%= 26.3  r(신뢰)=+0.233

[회피 프록시] Q84==9998 = 4.35%
  다양성: 회피군 4.38 vs 비회피군 6.26
  신뢰  : 회피군 2.875 vs 비회피군 3.070


## 7. 결론 — 차원 확정/축소 결정

| 차원 | 근거(본 EDA) | 결정 |
|------|------|------|
| **신뢰** | 배터리 결측·특수코드 0%, 문항 풍부 | ✅ **코어**. 단 ① Q91 **역코딩**, ② 요인분석으로 하위차원 정제 |
| **다양성** | 이용매체 21문항+Q84, 연령 강한 구배, 가중 시 디지털↑ | ✅ **코어**. ① **가중** ② **연령대 비닝** ③ **9998·NaN 분리** 필수 |
| 검증습관 | 구조적 결측 30~74%, 약한 상관(r≤0.23) | ⚠️ **보조 지표 강등**(코어 제외, 선택 트랙) |
| 회피 | Q84==9998 4.35%뿐, 직접문항 부재 | ⚠️ **이진 행동 플래그 강등**(차원 아님) |

### 다음 작업(전처리 설계 문서로 이관)
1. **특수코드 정책**: 9997/9998/9999 → 분석별 NaN 또는 별도 카테고리(행동변수)로 명시 분리.
2. **신뢰 차원**: Q91 역코딩 → 요인분석(신뢰성/공정성/언론인/사회신뢰 등 하위차원) → 신뢰지수.
3. **다양성 차원**: 게이트 이용매체 수 + Q84 기반 집중도(HHI), **가중·연령대 비닝** 적용.
4. **검증·회피**: 코어에서 제외, 보조 플래그(능동 구독/검색 지표·회피 플래그)로 부가.
5. 설계 산출 → `docs/design/preprocessing-design.md` 착수(본 노트북이 근거).

> 본 결론은 data-spec.md §5의 **옵션 (b) 신뢰·다양성 중심 축소**를 데이터로 확정한 것.